# 01 - SALT3 Initialize Vietnamese NeoBERT Embeddings

This notebook builds the SALT-initialized Vietnamese NeoBERT checkpoint using ViDeBERTa as the target PLM and NeoBERT as the source model. The output is a **global init artifact** that any CPT training run can reference.

Outputs are saved under `/content/drive/MyDrive/SALT3/init/<INIT_NAME>/`.

In [1]:
%%capture
!pip install -U transformers accelerate datasets safetensors sentencepiece tokenizers pandas scikit-learn matplotlib tqdm huggingface_hub fasttext-wheel deep_translator xformers

In [2]:
from pathlib import Path
import copy
import json
import os
import re
import shutil
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3') if Path('/content/drive/MyDrive').exists() else Path.cwd() / 'SALT3'
CODE_DIR = PROJECT_ROOT / 'code'
CODE_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE_DIR))
sys.path.insert(0, '/content')

from salt3_common import (
    configure_environment, copy_neobert_remote_files, embedding_fingerprint,
    ensure_dir, extract_embedding_weight, fit_embedding_to_decoder_map,
    load_model_safe, set_seed, target_unigram_logfreq_bias, write_json,
)

configure_environment()
set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Project root:', PROJECT_ROOT)

Mounted at /content/drive
Device: cuda
Project root: /content/drive/MyDrive/SALT3


## Configuration

Set `INIT_NAME` to a unique name for this init artifact. The output will be saved under `init/<INIT_NAME>/`. Multiple training runs can all reference the same init artifact.

In [3]:
# ── Init artifact name (change this when you re-init with different settings) ──
INIT_NAME = 'videberta_salt_init_v5_globalmap_freqbias'

# ── Model IDs ──
SOURCE_MODEL_ID  = 'chandar-lab/NeoBERT'        # source architecture
TARGET_MODEL_ID  = 'Fsoft-AIC/videberta-base'   # target PLM for Vietnamese embeddings
TRANSLATOR_PATH  = 'Helsinki-NLP/opus-mt-vi-en' # MarianMT vi→en

# ── Paths ──
INIT_DIR                = ensure_dir(PROJECT_ROOT / 'init' / INIT_NAME)
MODEL_DIR               = INIT_DIR / 'model'
PRUNED_TOKENIZER_DIR    = INIT_DIR / 'pruned_videberta_tokenizer'
ANCHOR_CSV              = INIT_DIR / 'salt_anchor_pairs.csv'
EMBEDDING_TENSOR        = INIT_DIR / 'init_vietnamese_embeddings.pt'
FASTTEXT_BIN            = str(INIT_DIR / 'cc.vi.300.bin')
FASTTEXT_URL            = 'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.vi.300.bin.gz'

# ── SALT / anchor mining settings ──
TARGET_TOKENIZER_USE_FAST = True
TARGET_VOCAB_SIZE         = None     # None = match source vocab size
TOP_K_ANCHORS             = 64
RIDGE                     = 1e-3
MIN_ANCHORS_FOR_LOCAL     = 8
BATCH_SIZE                = 64       # translation batch size for MarianMT

# ── Decoder (LM head) init ───────────────────────────────────────────────────
# NeoBERT's decoder is UNTIED and lives in post-RMSNorm hidden space. SALT's paper
# only ever transferred TIED heads, so its per-token lstsq is wrong here (tested
# ~15.0 step-0 MLM loss). We instead rebuild the head from a single GLOBAL
# embedding->decoder linear map fit over all source tokens (stable), applied to the
# SALT-projected embeddings. DECODER_WEIGHT_SCALE keeps the pre-CPT (English-shaped)
# decoder logits from swamping the frequency-prior bias, so step-0 loss ≈ Vietnamese
# unigram entropy (~7.3); CPT grows the head from there.
DECODER_WEIGHT_SCALE = 0.1
# Decoder bias = Vietnamese unigram log-frequency. NeoBERT's trained decoder bias is
# a large frequency prior (norm≈229); earlier inits wrongly zeroed it. Counted by
# streaming this many CulturaX-vi docs through the pruned tokenizer.
BIAS_COUNT_DOCS = 20000

print('Init name   :', INIT_NAME)
print('Init dir    :', INIT_DIR)
print('Model dir   :', MODEL_DIR)
print('Source model:', SOURCE_MODEL_ID)
print('Target PLM  :', TARGET_MODEL_ID)


Init name   : videberta_salt_init_v5_globalmap_freqbias
Init dir    : /content/drive/MyDrive/SALT3/init/videberta_salt_init_v5_globalmap_freqbias
Model dir   : /content/drive/MyDrive/SALT3/init/videberta_salt_init_v5_globalmap_freqbias/model
Source model: chandar-lab/NeoBERT
Target PLM  : Fsoft-AIC/videberta-base


In [4]:
# Optional: HF login for gated models (NeoBERT and ViDeBERTa are public)
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])

## Load Source and Target Models

In [5]:
source_tokenizer = AutoTokenizer.from_pretrained(SOURCE_MODEL_ID, trust_remote_code=True)
source_model = load_model_safe(SOURCE_MODEL_ID, model_cls=AutoModelForMaskedLM, device='cpu')
source_embeddings = extract_embedding_weight(source_model).float().cpu()
source_decoder_weight = source_model.decoder.weight.detach().float().cpu()
source_vocab = source_tokenizer.get_vocab()

if TARGET_VOCAB_SIZE is None:
    TARGET_VOCAB_SIZE = len(source_vocab)
print('NeoBERT embeddings:', tuple(source_embeddings.shape), 'vocab:', len(source_vocab))
print('NeoBERT decoder   :', tuple(source_decoder_weight.shape))

print('Loading full ViDeBERTa tokenizer/model...')
full_target_tokenizer = AutoTokenizer.from_pretrained(
    TARGET_MODEL_ID, use_fast=TARGET_TOKENIZER_USE_FAST, trust_remote_code=True
)
target_model = AutoModel.from_pretrained(TARGET_MODEL_ID, trust_remote_code=True).cpu()
target_embeddings = extract_embedding_weight(target_model).float().cpu()
target_emb_std = float(target_embeddings.std())
target_ln_means = [
    float(p.detach().float().mean())
    for name, p in target_model.named_parameters()
    if 'LayerNorm.weight' in name and 'mask_predictions' not in name
]
target_ln_mean = sum(target_ln_means) / len(target_ln_means) if target_ln_means else float('nan')
print('ViDeBERTa embeddings:', tuple(target_embeddings.shape), 'full tokenizer len:', len(full_target_tokenizer))
print(f'ViDeBERTa emb std : {target_emb_std:.6f}')
if target_ln_means:
    print(f'ViDeBERTa LN mean : {target_ln_mean:.6f}')
if target_emb_std < 0.12:
    raise RuntimeError(
        'ViDeBERTa word embeddings look untrained/random (std < 0.12). Abort SALT init and fix target-model loading first.'
    )

assert source_embeddings.shape[1] == target_embeddings.shape[1], 'Hidden sizes must match for this SALT setup.'

config.json:   0%|          | 0.00/928 [00:00<?, ?B/s]

model.py:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

rotary.py:   0%|          | 0.00/2.58k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- model.py
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/567M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: Fsoft-AIC/videberta-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ViDeBERTa embeddings: (128000, 768) full tokenizer len: 128000
ViDeBERTa emb std : 0.127559
ViDeBERTa LN mean : 0.472782


## Utility Functions for Anchor Mining

In [6]:
import re
import unicodedata


def is_full_word_and_clean(token: str, tokenizer_type: str) -> str | None:
    """Return the bare surface form if the token is a word-initial token, else None."""
    if tokenizer_type == 'videberta':
        # SentencePiece uses ▁ (U+2581) as word-start prefix
        if token.startswith('\u2581'):
            return token[1:]
        return None
    elif tokenizer_type == 'neobert':
        # NeoBERT uses BERT WordPiece: no prefix for word-initial tokens,
        # ## prefix for continuation tokens.
        if not token.startswith('##') and token.isalpha():
            return token
        return None
    return None


def is_strict_vietnamese_token(token):
    token = str(token).lower().strip()

    # 1. Reject non-native characters
    if re.search(r'[fjwz]', token):
        return False

    # 2. Length constraint
    if len(token) > 7:
        return False

    # 3. Exactly ONE contiguous vowel cluster
    vowels = r'aàáảãạăằắẳẵặâầấẩẫậeèéẻẽẹêềếểễệiìíỉĩịoòóỏõọôồốổỗộơờớởỡợuùúủũụưừứửữựyỳýỷỹỵ'
    vowel_clusters = re.findall(f'[{vowels}]+', token)
    if len(vowel_clusters) != 1:
        return False

    # 4. Strict Grammar & Orthography Constraints

    # 4a. 'k', 'gh', 'ngh' MUST precede i, e, ê, y
    front_vowels = r'[ieêyìíỉĩịèéẻẽẹềếểễệỳýỷỹỵ]'
    if re.match(r'^(k|gh|ngh)(?!' + front_vowels + ')', token):
        return False

    # 4b. 'c', 'g', 'ng' MUST NOT precede i, e, ê, y
    if re.match(r'^(c|g|ng)' + front_vowels, token):
        return False

    # 4c. 'q' MUST be followed by 'u'
    if re.match(r'^q[^uùúủũụ]', token):
        return False

    # 4d. 'p' cannot be an onset alone (filters pô, pin, etc.)
    if re.match(r'^p[^h]', token) or token == 'p':
        return False

    # 4e. Invalid foreign vowel combinations
    if re.search(r'(yu|io|ea|ee|oo|ii|uu|eon|ion|ian)', token):
        return False

    # 4f. Incomplete vowels MUST have a coda (filters nhắ, tầ, nghiê, viê)
    if re.search(r'[ăằắẳẵặâầấẩẫậ]$', token):
        return False
    if re.search(r'([iìíỉĩịyỳýỷỹỵ][êềếểễệ]|[uùúủũụ][ôồốổỗộ]|[ưừứửữự][ơờớởỡợ])$', token):
        return False

    # 5. Extract Prefix and Suffix
    parts = re.split(f'[{vowels}]+', token)
    if len(parts) == 2:
        prefix = parts[0]
        suffix = parts[1]

        valid_prefixes = [
            '', 'b', 'c', 'ch', 'd', 'đ', 'g', 'gh', 'gi', 'h', 'k', 'kh',
            'l', 'm', 'n', 'ng', 'ngh', 'nh', 'ph', 'q', 'qu', 'r',
            's', 't', 'th', 'tr', 'v', 'x',
        ]
        valid_suffixes = ['', 'c', 'ch', 'm', 'n', 'ng', 'nh', 'p', 't']

        if prefix not in valid_prefixes:
            return False
        if suffix not in valid_suffixes:
            return False

        # 6. The Tone/Coda Rule (Filters sec, nhac, mit, tit)
        # Words ending in c, ch, p, t MUST have a Sắc (á) or Nặng (ạ) tone.
        if suffix in ['c', 'ch', 'p', 't']:
            sac_nang_vowels = r'[áấắéếíóốớúứýạậặẹệịọộợụựỵ]'
            if not re.search(sac_nang_vowels, token):
                return False

    # 7. Hardcoded filter for valid structures that are noise
    known_noise = {
        'them', 'cheng', 'ken', 'gợ', 'bit', 'liu', 'đái', 'xị', 'thố',
        'vùn', 'duẫn', 'oai', 'duẫn', 'om', 'xiêm', 'cang', 'liu',
        'rè', 'ngan', 'hỉ', 'ngoa', 'thá', 'rá', 'meo', 'dõ', 'hoi',
        'bện', 'phạn', 'mít', 'xoan', 'vự', 'lán', 'chọ', 'lợ', 'dũ',
        'lia', 'thoạ', 'nhụ', 'nhạn', 'ách', 'nim', 'tiệp', 'chiểu',
        'trư', 'trảng', '',
    }
    if token in known_noise:
        return False

    return True


def sparsemax(logits: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Project logits onto the probability simplex (Sparsemax), Martins & Astudillo 2016."""
    sorted_logits, _ = torch.sort(logits, dim=dim, descending=True)
    cum_sums = torch.cumsum(sorted_logits, dim=dim)
    k = torch.arange(1, logits.size(dim) + 1, device=logits.device, dtype=logits.dtype).expand_as(logits)
    k_z = (1 + k * sorted_logits > cum_sums).sum(dim=dim, keepdim=True).float()
    tau = (torch.gather(cum_sums, dim, (k_z - 1).long()) - 1) / k_z
    return torch.clamp(logits - tau, min=0.0)


print('Utility functions defined.')


Utility functions defined.


## Prune ViDeBERTa Tokenizer

Prune the ViDeBERTa SentencePiece vocab down to `TARGET_VOCAB_SIZE` tokens (keeping specials + top-scored normals). Build `new_to_old` / `old_to_new` maps for the embedding projection step.

In [7]:
tok_json_path = hf_hub_download(repo_id=TARGET_MODEL_ID, filename='tokenizer.json')
with open(tok_json_path, 'r', encoding='utf-8') as f:
    tok_data = json.load(f)

full_vocab = tok_data['model']['vocab']
added_token_ids = {entry['id'] for entry in tok_data.get('added_tokens', [])}
all_special_ids = set(full_target_tokenizer.all_special_ids) | added_token_ids

special_pieces = []
normal_pieces = []
for old_id, (piece, score) in enumerate(full_vocab):
    if old_id in all_special_ids or score == 0.0:
        special_pieces.append((old_id, piece, score))
    else:
        normal_pieces.append((old_id, piece, score))
normal_pieces.sort(key=lambda x: x[2], reverse=True)

n_normal_to_keep = TARGET_VOCAB_SIZE - len(special_pieces)
assert n_normal_to_keep > 0, f'Too many special tokens ({len(special_pieces)}) for target size {TARGET_VOCAB_SIZE}'
kept_pieces = sorted(special_pieces, key=lambda x: x[0]) + normal_pieces[:n_normal_to_keep]

new_to_old = {}
old_to_new = {}
for new_id, (old_id, _, _) in enumerate(kept_pieces):
    new_to_old[new_id] = old_id
    old_to_new[old_id] = new_id

pruned_tok_data = copy.deepcopy(tok_data)
pruned_tok_data['model']['vocab'] = [[piece, score] for _, piece, score in kept_pieces]
old_unk_id = tok_data['model'].get('unk_id', 0)
pruned_tok_data['model']['unk_id'] = old_to_new.get(old_unk_id, 0)

new_added_tokens = []
for entry in tok_data.get('added_tokens', []):
    if entry['id'] in old_to_new:
        new_entry = dict(entry)
        new_entry['id'] = old_to_new[entry['id']]
        new_added_tokens.append(new_entry)
pruned_tok_data['added_tokens'] = sorted(new_added_tokens, key=lambda x: x['id'])

PRUNED_TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
with (PRUNED_TOKENIZER_DIR / 'tokenizer.json').open('w', encoding='utf-8') as f:
    json.dump(pruned_tok_data, f, ensure_ascii=False, indent=2)
for filename in ('tokenizer_config.json', 'special_tokens_map.json'):
    src = hf_hub_download(repo_id=TARGET_MODEL_ID, filename=filename)
    shutil.copy(src, PRUNED_TOKENIZER_DIR / filename)

target_tokenizer = AutoTokenizer.from_pretrained(PRUNED_TOKENIZER_DIR, trust_remote_code=True)
target_vocab = {piece: new_id for new_id, (_, piece, _) in enumerate(kept_pieces)}
target_vocab_size = len(target_tokenizer)
assert target_vocab_size == len(target_vocab)

special_token_pairs = []
seen_target_ids = set()
for key in ('pad_token', 'unk_token', 'cls_token', 'sep_token', 'mask_token', 'bos_token', 'eos_token'):
    target_token = getattr(target_tokenizer, key, None)
    source_token = getattr(source_tokenizer, key, None)
    if target_token is None or source_token is None:
        continue
    target_id = target_vocab.get(target_token)
    source_id = source_vocab.get(source_token)
    if target_id is None or source_id is None or target_id in seen_target_ids:
        continue
    special_token_pairs.append({
        'key': key,
        'target_token': target_token,
        'target_id': target_id,
        'source_token': source_token,
        'source_id': source_id,
    })
    seen_target_ids.add(target_id)

special_target_tokens = {row['target_token'] for row in special_token_pairs}
special_target_ids = {row['target_id'] for row in special_token_pairs}
assert target_tokenizer.mask_token_id in special_target_ids, 'Mask token row must be preserved explicitly.'

print('Original ViDeBERTa vocab:', len(full_vocab))
print('Pruned ViDeBERTa vocab  :', target_vocab_size)
print('NeoBERT vocab           :', len(source_vocab))
print('Tokenizer pad id        :', target_tokenizer.pad_token_id)
print('Protected special rows  :', len(special_target_ids))
for row in special_token_pairs:
    print(f"  {row['key']}: {row['target_token']} (target {row['target_id']}) <- {row['source_token']} (source {row['source_id']})")
ids = target_tokenizer.encode('Xin chào Việt Nam')
print('Sanity ids:', ids, 'max:', max(ids))
assert max(ids) < target_vocab_size

model.safetensors:   0%|          | 0.00/567M [00:00<?, ?B/s]

Original ViDeBERTa vocab: 128000
Pruned ViDeBERTa vocab  : 30522
NeoBERT vocab           : 30522
Tokenizer pad id        : 0
Protected special rows  : 5
  pad_token: [PAD] (target 0) <- [PAD] (source 0)
  unk_token: [UNK] (target 3) <- [UNK] (source 100)
  cls_token: [CLS] (target 1) <- [CLS] (source 101)
  sep_token: [SEP] (target 2) <- [SEP] (source 102)
  mask_token: [MASK] (target 4) <- [MASK] (source 103)
Sanity ids: [1, 1049, 1648, 358, 627, 2] max: 1648


## Build Anchor Pairs (3-Tier Translation Mining)

Anchors are token pairs where a pruned ViDeBERTa token and a NeoBERT token are semantically equivalent. Three sources are used:

1. **Shared numbers** – digit tokens present in both vocabularies.
2. **Verified shared surface forms** – tokens with the same surface form, confirmed via MarianMT back-translation (vi→en→vi must roundtrip).
3. **3-tier translation mining** – systematic vi→en translation:
   - *Tier 1*: back-translation verification (Google + MarianMT), single-word English output only.
   - *Tier 2*: compound Vietnamese words translated to a single English word via Google Translate.
   - *Tier 3*: remaining single Vietnamese words, blacklist-filtered, capped at 500 pairs.

The master `anchor_map` (vi_token → neo_token) drives both the FastText projection and the anchor re-injection step.


In [8]:
# Aliases for variables renamed during refactor
pruned_vi_vocab = target_vocab
neo_vocab = source_vocab
OUTPUT_DIR = str(INIT_DIR)

# ── 5a. Build word lists ─────────────────────────────────────────────────────
vi_full_words: dict[str, str] = {}
neo_full_words: dict[str, str] = {}

for token in pruned_vi_vocab:
    c = is_full_word_and_clean(token, "videberta")
    if c:
        parts = c.split('_')
        if len(parts) > 1 and all(part and part[0].isupper() for part in parts):
            continue
        if not all(part.isalpha() for part in parts):
            continue
        norm_key = " ".join(parts).lower()
        vi_full_words[norm_key] = token

for token in neo_vocab:
    c = is_full_word_and_clean(token, "neobert")
    if c and c.isalpha():
        neo_full_words[c.lower()] = token

vi_strict  = {k: v for k, v in vi_full_words.items()  if len(k.replace(" ", "")) > 3}
neo_strict = {k: v for k, v in neo_full_words.items() if len(k) > 3}
surface_candidates = set(vi_strict) & set(neo_strict)

# ── 5b. Shared numbers ───────────────────────────────────────────────────────
vi_numbers  = {}
neo_numbers = {}

for token in pruned_vi_vocab:
    c = token.replace('▁', '')
    if c.isdigit():
        vi_numbers[c] = token

for token in neo_vocab:
    c = token.replace('Ġ', '')
    if c.isdigit():
        neo_numbers[c] = token

shared_numbers = set(vi_numbers) & set(neo_numbers)

df_numbers = pd.DataFrame([
    {"shared_token": n, "videberta_token": vi_numbers[n], "neobert_token": neo_numbers[n]}
    for n in sorted(shared_numbers, key=int)
])
df_numbers.to_csv(os.path.join(OUTPUT_DIR, "shared_numbers.csv"), index=False)
print(f"Shared numbers saved : {len(df_numbers)}")

# ── 5c. Load translation models ──────────────────────────────────────────────
from transformers import MarianTokenizer, MarianMTModel
from deep_translator import GoogleTranslator
import time

print("Loading MarianMT model (Helsinki-NLP/opus-mt-vi-en)...")
marian_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-vi-en")
marian_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-vi-en").to(DEVICE)

def translate_marian(texts: list[str]) -> list[str]:
    inputs = marian_tok(texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    out = marian_model.generate(**inputs)
    return marian_tok.batch_decode(out, skip_special_tokens=True)

def google_translate_batch(texts: list[str], delay: float = 0.2) -> list[str]:
    translator = GoogleTranslator(source='vi', target='en')
    results = []
    for t in texts:
        try:
            translated = translator.translate(t)
            results.append(translated if translated else "")
        except:
            results.append("")
        time.sleep(delay)
    return results

# ── 5d. Verified shared anchors (using MarianMT) ──────────────────────────────
print(f"Verifying {len(surface_candidates)} surface‑form anchor candidates…")
candidates_list = list(surface_candidates)
verified_anchors = []

for i in tqdm(range(0, len(candidates_list), BATCH_SIZE)):
    batch = candidates_list[i : i + BATCH_SIZE]
    translations = translate_marian(batch)
    for original, translated in zip(batch, translations):
        if translated.lower().strip() == original:
            verified_anchors.append({
                "shared_token": original,
                "videberta_token": vi_strict[original],
                "neobert_token": neo_strict[original],
            })

df_verified = pd.DataFrame(verified_anchors)
df_verified.to_csv(os.path.join(OUTPUT_DIR, "verified_shared_anchors.csv"), index=False)
print(f"Verified shared anchors saved : {len(df_verified)}")

# ═══════════════════════════════════════════════════════════════════════════════
# 5e. 3‑tier translation mining
# ═══════════════════════════════════════════════════════════════════════════════

vi_translation_words = {
    k: v for k, v in vi_full_words.items()
    if all(is_strict_vietnamese_token(part) for part in k.split())
}
vi_words_list = list(vi_translation_words.keys())
neo_vocab_set = set(neo_full_words.keys())

# Blacklist cho Tier 3
blacklist = {"em", "cả", "thứ", "cấp", "cuộc", "bá", "nhu", "ngữ", "ngân", "bồi", "lơ", "đồn", "bề", "đã","bị", "sự","cô","tới",
             "cái", "đều","do","ấy", "thế", "các", "này", "để","những","được", "chín","phạm","thể", "cục", "vặt", "hợp","lực"
             , "thông", "xét","vết", "động","thiệt", "nha", "âu", "minh", "nỗi","trung","kỹ","cơn","bất","nội","chi","kém","tiến",
             "hạ", "nghiệm", "hệ", "nhập", "hội", "chàng", "phân", "tầm", "phát", "ánh", "thời", "ngôi", "trời","quý","bức","vẻ",
             "hãng", "hiện","thân", "công", "bản", "đấy", "căn", "pháp","cứ","đồ", "kia", "bé", "thành","đạt","càng","suy","cà",
             "yên", "tốn", "tỏ", "kín","truy","diện","hữu","trống","trúng","xuyên", "thượng","tớ","lao","hôm","vốn","chẳng"}

# Helper: back‑translation
def back_translation_test_single(vi_word, translator_func, neo_set):
    en_list = translator_func([vi_word])
    if not en_list or not en_list[0]:
        return None, False
    en = en_list[0].strip().lower()
    if not en or ' ' in en:
        return None, False
    try:
        back_vi = GoogleTranslator(source='en', target='vi').translate(en)
    except:
        return None, False
    if not back_vi:
        return None, False
    back_vi = back_vi.strip().lower()
    if back_vi == vi_word and en != vi_word:
        if en in neo_set:
            return en, True
    return None, False

new_pairs = []
processed_words = set()
tier1_count = 0
tier2_count = 0
tier3_count = 0

# ---- Tier 1: Back‑translation (Google + Marian) ----
print("Tier 1: Back‑translation (chỉ giữ en đơn)...")
BATCH_T1 = 64
for i in tqdm(range(0, len(vi_words_list), BATCH_T1), desc="Tier 1"):
    batch = vi_words_list[i:i+BATCH_T1]
    for vi in batch:
        if vi in processed_words:
            continue
        # Google first
        en_word, ok = back_translation_test_single(vi, google_translate_batch, neo_vocab_set)
        if ok:
            new_pairs.append({
                'vi_clean_token': vi,
                'en_clean_token': en_word,
                'videberta_token': vi_translation_words[vi],
                'neobert_token': neo_full_words[en_word]
            })
            processed_words.add(vi)
            tier1_count += 1
            continue
        # Then Marian
        en_word, ok = back_translation_test_single(vi, translate_marian, neo_vocab_set)
        if ok:
            new_pairs.append({
                'vi_clean_token': vi,
                'en_clean_token': en_word,
                'videberta_token': vi_translation_words[vi],
                'neobert_token': neo_full_words[en_word]
            })
            processed_words.add(vi)
            tier1_count += 1
print(f"  Tier 1 added {tier1_count} pairs.")

# ---- Tier 2: Compound words (contain space) ----
print("Tier 2: Compound words (direct Google Translate, chỉ giữ en đơn)...")
remaining_words = [w for w in vi_words_list if w not in processed_words]
for vi in tqdm(remaining_words, desc="Tier 2"):
    if ' ' in vi:
        try:
            en = GoogleTranslator(source='vi', target='en').translate(vi)
        except:
            continue
        if en:
            en_word = en.strip().lower()
            if ' ' in en_word:
                continue
            if en_word in neo_vocab_set:
                new_pairs.append({
                    'vi_clean_token': vi,
                    'en_clean_token': en_word,
                    'videberta_token': vi_translation_words[vi],
                    'neobert_token': neo_full_words[en_word]
                })
                processed_words.add(vi)
                tier2_count += 1
print(f"  Tier 2 added {tier2_count} pairs.")

# ---- Tier 3: Single words, blacklist filtered, max 500 ----
print("Tier 3: Single words (blacklist filtered, max 500)...")
remaining_words = [w for w in vi_words_list if w not in processed_words]
single_words = [w for w in remaining_words if ' ' not in w and w not in blacklist]
MAX_TIER3 = 500
for vi in tqdm(single_words, desc="Tier 3"):
    if tier3_count >= MAX_TIER3:
        break
    try:
        en = GoogleTranslator(source='vi', target='en').translate(vi)
    except:
        continue
    if en:
        en_word = en.strip().lower()
        if ' ' in en_word:
            continue
        if en_word in neo_vocab_set:
            new_pairs.append({
                'vi_clean_token': vi,
                'en_clean_token': en_word,
                'videberta_token': vi_translation_words[vi],
                'neobert_token': neo_full_words[en_word]
            })
            processed_words.add(vi)
            tier3_count += 1
print(f"  Tier 3 added {tier3_count} pairs (capped at {MAX_TIER3}).")

# Save the new translation pairs
df_pairs = pd.DataFrame(new_pairs)
csv_pairs_path = os.path.join(OUTPUT_DIR, "verified_translation_pairs.csv")
df_pairs.to_csv(csv_pairs_path, index=False)
print(f"Created {csv_pairs_path} with {len(df_pairs)} pairs (3‑tier mining).")

# ═══════════════════════════════════════════════════════════════════════════════
# 5f. Build master anchor_map (vi original token → neo original token)
# ═══════════════════════════════════════════════════════════════════════════════
anchor_frames = []
for path, vi_col, neo_col in [
    (os.path.join(OUTPUT_DIR, "verified_shared_anchors.csv"), "videberta_token", "neobert_token"),
    (os.path.join(OUTPUT_DIR, "shared_numbers.csv"), "videberta_token", "neobert_token"),
    (csv_pairs_path, "videberta_token", "neobert_token"),
]:
    if os.path.exists(path):
        df = pd.read_csv(path, dtype=str)
        anchor_frames.append(df[[vi_col, neo_col]].rename(columns={vi_col: "vi", neo_col: "neo"}))
    else:
        print(f"Warning: {path} not found")

if anchor_frames:
    df_master = pd.concat(anchor_frames, ignore_index=True).drop_duplicates()
    anchor_map: dict[str, str] = dict(zip(df_master["vi"], df_master["neo"]))
else:
    anchor_map = {}

print(f"\nTotal anchor pairs (all sources): {len(anchor_map)}")

Shared numbers saved : 580
Loading MarianMT model (Helsinki-NLP/opus-mt-vi-en)...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

...

  Tier 1 added 1537 pairs.
Tier 2: Compound words (direct Google Translate, chỉ giữ en đơn)...


Tier 2:   0%|          | 0/4087 [00:00<?, ?it/s]

  Tier 2 added 1261 pairs.
Tier 3: Single words (blacklist filtered, max 500)...


Tier 3:   0%|          | 0/2134 [00:00<?, ?it/s]

  Tier 3 added 500 pairs (capped at 500).
Created /content/drive/MyDrive/SALT3/init/videberta_salt_init_v5_globalmap_freqbias/verified_translation_pairs.csv with 3298 pairs (3‑tier mining).

Total anchor pairs (all sources): 4990


## Download & Load FastText Vietnamese Model

Pre-compute normalised FastText embeddings for every anchor token. These are used as the neighbourhood signal in the SALT projection step.


In [9]:
import fasttext
import gzip
import urllib.request

gz_path = FASTTEXT_BIN + '.gz'

if not os.path.exists(FASTTEXT_BIN):
    print(f'Downloading {FASTTEXT_URL} ...')
    urllib.request.urlretrieve(FASTTEXT_URL, gz_path)
    print('Decompressing...')
    with gzip.open(gz_path, 'rb') as f_in, open(FASTTEXT_BIN, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(gz_path)
    print('Done.')
else:
    print(f'{FASTTEXT_BIN} already present, skipping download.')

ft = fasttext.load_model(FASTTEXT_BIN)
print('FastText model loaded.')

# Special rows are copied directly from NeoBERT and must never go through SALT.
shared_vi_tokens = [t for t in anchor_map.keys() if t not in special_target_tokens]
non_shared_vi_tokens = [t for t in target_vocab if t not in anchor_map and t not in special_target_tokens]

def ft_vector(token: str) -> np.ndarray:
    """Return the FastText vector for a (possibly ▁-prefixed) token."""
    clean = token.replace('\u2581', '')
    return ft.get_word_vector(clean)

anchor_ft_matrix = np.array([ft_vector(t) for t in shared_vi_tokens], dtype=np.float32)
anchor_ft_tensor = F.normalize(
    torch.tensor(anchor_ft_matrix, dtype=torch.float32, device=DEVICE), p=2, dim=1
)

print(f'Anchor FastText matrix : {anchor_ft_tensor.shape}')
print(f'Protected special rows : {len(special_target_ids)}')
print(f'Non-shared tokens      : {len(non_shared_vi_tokens)}')


Decompressing...
Done.
FastText model loaded.
Anchor FastText matrix : torch.Size([4990, 300])
Protected special rows : 5
Non-shared tokens      : 25527


## SALT Projection (Embeddings + Decoder)

Per the SALT paper, input embeddings are projected from ViDeBERTa space into NeoBERT space. Since NeoBERT has an independent (untied) decoder, we also project NeoBERT's decoder rows through the same anchor mapping.

For each non-shared pruned ViDeBERTa token:
1. Compute cosine similarity vs every anchor's FastText vector.
2. Apply **Sparsemax** → sparse support over anchors.
3. **If k >= MIN_ANCHORS_FOR_LOCAL (8)**: Solve local least-squares $E_t' \to E_s'$ for embeddings and $E_t' \to D_s'$ for decoder. Enough anchor points for lstsq to be meaningful.
4. **If k < MIN_ANCHORS_FOR_LOCAL**: Use sparsemax-weighted average of anchor NeoBERT embeddings/decoder rows. With too few anchors, lstsq is wildly underdetermined (k equations for 768 unknowns) and produces unstable projections.

For shared anchor tokens: direct copy from NeoBERT (both embedding and decoder rows).

In [10]:
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

source_emb_dev = source_embeddings.to(DEVICE)
target_emb_dev = target_embeddings.to(DEVICE)

d_neo = source_emb_dev.shape[1]
source_mean = source_emb_dev.mean()
source_std = source_emb_dev.std()

new_embeddings = torch.zeros(
    (target_vocab_size, d_neo), dtype=torch.float32, device=DEVICE
)

# Projection quality tracking
n_lstsq = 0
n_weighted_avg = 0
n_random_fallback = 0
anchor_counts = []

print(f'Projecting {len(non_shared_vi_tokens)} non-shared token EMBEDDINGS (SALT per-token lstsq)...')
print(f'  MIN_ANCHORS_FOR_LOCAL={MIN_ANCHORS_FOR_LOCAL}: lstsq when k >= {MIN_ANCHORS_FOR_LOCAL}, weighted-avg otherwise')
print('  Decoder (LM head) is built separately from a global emb->dec map in the save cell.')

for vi_token in tqdm(non_shared_vi_tokens):
    vi_new_id = target_vocab[vi_token]
    vi_old_id = new_to_old[vi_new_id]

    if vi_old_id >= target_emb_dev.shape[0]:
        new_embeddings[vi_new_id] = torch.normal(source_mean, source_std, size=(d_neo,), device=DEVICE)
        n_random_fallback += 1
        continue

    target_emb = target_emb_dev[vi_old_id].unsqueeze(0)

    ft_vec = torch.tensor(ft_vector(vi_token), dtype=torch.float32, device=DEVICE)
    ft_vec = F.normalize(ft_vec.unsqueeze(0), p=2, dim=1)
    sim = torch.matmul(ft_vec, anchor_ft_tensor.T).squeeze(0)

    sparse_support = sparsemax(sim, dim=0)
    nz_indices = torch.nonzero(sparse_support).squeeze(1)

    if len(nz_indices) == 0:
        new_embeddings[vi_new_id] = torch.normal(source_mean, source_std, size=(d_neo,), device=DEVICE)
        n_random_fallback += 1
        continue

    k = len(nz_indices)
    anchor_counts.append(k)
    nz_idx_list = nz_indices.tolist()
    sel_vi = [shared_vi_tokens[j] for j in nz_idx_list]
    sel_neo = [anchor_map[v] for v in sel_vi]

    E_t = torch.stack([target_emb_dev[new_to_old[target_vocab[v]]] for v in sel_vi])
    E_s = torch.stack([source_emb_dev[source_vocab[n]] for n in sel_neo])

    if k >= MIN_ANCHORS_FOR_LOCAL:
        # SALT Eq. 3: local least-squares map from donor embedding space to source space
        # SALT Eq.3 min-norm solution X = pinv(E_t) @ E_s. pinv is well-defined
        # for the underdetermined case here (k anchors < 768 dims), unlike CUDA
        # lstsq whose underdetermined behavior is driver/version-dependent.
        X_emb = torch.linalg.pinv(E_t) @ E_s
        new_embeddings[vi_new_id] = torch.matmul(target_emb, X_emb).squeeze(0)
        n_lstsq += 1
    else:
        # Too few anchors for a stable lstsq: convex (sparsemax-weighted) average
        weights = sparse_support[nz_indices].unsqueeze(1)
        w_sum = weights.sum()
        new_embeddings[vi_new_id] = (E_s * weights).sum(0) / w_sum
        n_weighted_avg += 1

print('Injecting anchor and special-token embedding rows...')
for vi_token, neo_token in anchor_map.items():
    new_embeddings[target_vocab[vi_token]] = source_emb_dev[source_vocab[neo_token]]
for row in special_token_pairs:
    new_embeddings[row['target_id']] = source_emb_dev[row['source_id']]

new_embeddings = new_embeddings.cpu()

anchor_counts_t = torch.tensor(anchor_counts, dtype=torch.float32) if anchor_counts else torch.zeros(1)
print(f'\nEmbedding projection complete.')
print(f'  Embedding shape: {tuple(new_embeddings.shape)}, mean norm: {float(new_embeddings.norm(dim=1).mean()):.4f}')
print(f'\n── Projection method breakdown ──')
print(f'  lstsq (k >= {MIN_ANCHORS_FOR_LOCAL})  : {n_lstsq:,}')
print(f'  weighted-avg (k < {MIN_ANCHORS_FOR_LOCAL}): {n_weighted_avg:,}')
print(f'  random fallback    : {n_random_fallback:,}')
print(f'  Anchor count stats : mean={anchor_counts_t.mean():.1f}, median={anchor_counts_t.median():.0f}, min={anchor_counts_t.min():.0f}, max={anchor_counts_t.max():.0f}')

del source_emb_dev, target_emb_dev
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Projecting 25527 non-shared token EMBEDDINGS (SALT per-token lstsq)...
  MIN_ANCHORS_FOR_LOCAL=8: lstsq when k >= 8, weighted-avg otherwise
  Decoder (LM head) is built separately from a global emb->dec map in the save cell.


  0%|          | 0/25527 [00:00<?, ?it/s]

Injecting anchor and special-token embedding rows...

Embedding projection complete.
  Embedding shape: (30522, 768), mean norm: 0.5464

── Projection method breakdown ──
  lstsq (k >= 8)  : 25,457
  weighted-avg (k < 8): 70
  random fallback    : 0
  Anchor count stats : mean=49.2, median=26, min=3, max=4990


## Save Init Artifact

Save the SALT-initialized model under `init/<INIT_NAME>/model/`:
- **Embeddings**: SALT-projected from ViDeBERTa, per-token normalized to NeoBERT mean norm
- **Decoder**: Independently projected from NeoBERT's decoder via same anchor mapping, per-token normalized
- **Tokenizer**: Pruned ViDeBERTa (30,522 tokens)
- **Config**: `tie_word_embeddings=False` (matches NeoBERT's original untied architecture)

In [11]:
# ── 0. Anchor and special-token EMBEDDING re-injection: exact NeoBERT fidelity ─
print('── Re-injecting anchor/special embedding rows from live NeoBERT ──')
anchor_vi_ids: set[int] = set()
for vi_token, neo_token in anchor_map.items():
    vi_id = target_vocab.get(vi_token)
    neo_id = source_vocab.get(neo_token)
    if vi_id is not None and neo_id is not None:
        new_embeddings[vi_id] = source_embeddings[neo_id].cpu()
        anchor_vi_ids.add(vi_id)
for row in special_token_pairs:
    new_embeddings[row['target_id']] = source_embeddings[row['source_id']].cpu()
print(f'  {len(anchor_vi_ids)} anchor rows injected')
print(f'  {len(special_target_ids)} special rows injected')

# ── 1. Embedding norm calibration: per-token to NeoBERT mean (direction kept) ──
neo_emb_norms = source_embeddings.norm(dim=1)
neo_emb_mean_norm = neo_emb_norms[neo_emb_norms > 1e-6].mean().item()

non_anchor_mask = torch.ones(target_vocab_size, dtype=torch.bool)
for aid in anchor_vi_ids:
    non_anchor_mask[aid] = False
for sid in special_target_ids:
    non_anchor_mask[sid] = False
if target_tokenizer.pad_token_id is not None and target_tokenizer.pad_token_id < target_vocab_size:
    non_anchor_mask[target_tokenizer.pad_token_id] = False

emb_before = new_embeddings.norm(dim=1)[non_anchor_mask]
print(f'\n── Embedding norm calibration ──')
print(f'  NeoBERT emb mean norm         : {neo_emb_mean_norm:.4f}')
print(f'  SALT non-anchor mean (before) : {emb_before[emb_before > 1e-6].mean().item():.4f}')
print(f'  SALT non-anchor max  (before) : {emb_before.max().item():.4f}')

cur_emb_norms = new_embeddings[non_anchor_mask].norm(dim=1, keepdim=True)
new_embeddings[non_anchor_mask] *= neo_emb_mean_norm / cur_emb_norms.clamp(min=1e-8)

emb_after = new_embeddings.norm(dim=1)[non_anchor_mask]
print(f'  SALT non-anchor mean (after)  : {emb_after.mean().item():.4f}')
print(f'  SALT non-anchor std  (after)  : {emb_after.std().item():.6f}')

# ── 2. Decoder (LM head) = GLOBAL emb->dec map applied to SALT embeddings ──────
# Per-token decoder projection (embedding-space -> decoder-space via k<=64 anchors)
# is geometrically unsound for an untied head and tested worse (~15.0 vs ~10.7
# step-0 MLM loss). One global map fit over ALL source tokens is stable and stays
# consistent with the projected embeddings.
print(f'\n── Decoder init: global NeoBERT emb->dec map ──')
emb_to_dec_map, map_residual = fit_embedding_to_decoder_map(source_embeddings, source_decoder_weight)
print(f"  global emb->dec relative residual on NeoBERT: {map_residual:.3f} "
      f"({'OK — linear head exists' if map_residual < 0.7 else 'HIGH — consider tying'})")
new_decoder = new_embeddings.float() @ emb_to_dec_map
# Down-scale so the pre-CPT (still English-shaped) decoder logits do not swamp the
# frequency-prior bias; CPT grows the head. ≈0.1 -> step-0 loss ≈ unigram entropy.
new_decoder *= DECODER_WEIGHT_SCALE
print(f'  decoder weight scale          : {DECODER_WEIGHT_SCALE}')
print(f'  decoder mean row norm         : {float(new_decoder.norm(dim=1).mean()):.4f}')

# ── 3. Decoder bias = Vietnamese unigram log-frequency (prior NeoBERT keeps) ───
print(f'\n── Decoder bias: Vietnamese unigram log-frequency ({BIAS_COUNT_DOCS} CulturaX docs) ──')
print(f'  NeoBERT decoder bias norm (was zeroed before): {source_model.decoder.bias.norm().item():.2f}')
new_decoder_bias, _bias_counts = target_unigram_logfreq_bias(
    target_tokenizer, target_vocab_size, num_docs=BIAS_COUNT_DOCS, lang='vi',
)
print(f'  VI freq bias range            : [{new_decoder_bias.min():.2f}, {new_decoder_bias.max():.2f}]')
print(f'  tokens counted                : {int(_bias_counts.sum()):,}')

# ── 4. Save raw embedding tensor ──────────────────────────────────────────────
torch.save(new_embeddings.cpu(), EMBEDDING_TENSOR)
print(f'\nEmbedding tensor saved: {EMBEDDING_TENSOR}')

# ── 5. Attach new embedding + decoder to NeoBERT ──────────────────────────────
print('Injecting embeddings and decoder into NeoBERT architecture...')
new_encoder = nn.Embedding(
    target_vocab_size,
    source_model.config.hidden_size,
    padding_idx=target_tokenizer.pad_token_id,
)
new_encoder.weight.data.copy_(new_embeddings.cpu())
source_model.model.encoder = new_encoder

source_model.decoder = nn.Linear(source_model.config.hidden_size, target_vocab_size)
source_model.decoder.weight.data.copy_(new_decoder.cpu())
source_model.decoder.bias.data.copy_(new_decoder_bias.cpu())
assert torch.isfinite(source_model.decoder.weight).all(), 'Non-finite decoder weights!'
assert torch.isfinite(source_model.decoder.bias).all(), 'Non-finite decoder bias!'
print(f'LM head decoder: {tuple(source_model.decoder.weight.shape)}')

source_model.config.vocab_size = target_vocab_size
source_model.config.pad_token_id = target_tokenizer.pad_token_id
source_model.config.tie_word_embeddings = False

# ── 6. Save model + pruned tokenizer ─────────────────────────────────────────
MODEL_DIR.mkdir(parents=True, exist_ok=True)
source_model.save_pretrained(MODEL_DIR)
target_tokenizer.save_pretrained(MODEL_DIR)
copy_neobert_remote_files(MODEL_DIR, SOURCE_MODEL_ID)
print(f'Model + tokenizer saved: {MODEL_DIR}')

# ── 6b. Safetensors integrity check ──────────────────────────────────────────
print('\n── Safetensors integrity check ──')
from safetensors import safe_open
st_path = MODEL_DIR / 'model.safetensors'
if st_path.exists():
    with safe_open(str(st_path), framework='pt', device='cpu') as f:
        keys = list(f.keys())
        has_emb = 'model.encoder.weight' in keys
        has_dec = 'decoder.weight' in keys
        print(f'  Keys in file: {len(keys)}')
        print(f'  model.encoder.weight present: {has_emb}')
        print(f'  decoder.weight present      : {has_dec}')
        if has_emb:
            raw_emb = f.get_tensor('model.encoder.weight')
            st_diff = (raw_emb - new_embeddings.cpu()).abs().max().item()
            print(f"  Max |safetensors - SALT emb| = {st_diff:.2e}  {'✓ OK' if st_diff < 1e-5 else '✗ WRONG'}")
        if has_dec:
            raw_dec = f.get_tensor('decoder.weight')
            dec_diff = (raw_dec - new_decoder.cpu()).abs().max().item()
            print(f"  Max |safetensors - decoder|  = {dec_diff:.2e}  {'✓ OK' if dec_diff < 1e-5 else '✗ WRONG'}")

# ── 7. Save salt_config.json ──────────────────────────────────────────────────
write_json(INIT_DIR / 'salt_config.json', {
    'init_name': INIT_NAME,
    'source_model': SOURCE_MODEL_ID,
    'target_model': TARGET_MODEL_ID,
    'target_vocab_size': target_vocab_size,
    'original_target_vocab_size': len(full_vocab),
    'target_embedding_rows_available': int(target_embeddings.shape[0]),
    'anchor_pairs': int(len(anchor_map)),
    'embedding_projection': f'salt_lstsq_k>={MIN_ANCHORS_FOR_LOCAL}_else_weighted_avg',
    'projection_stats': {'lstsq': n_lstsq, 'weighted_avg': n_weighted_avg, 'random_fallback': n_random_fallback},
    'norm_calibration': 'per_token_to_neobert_mean',
    'decoder_init': 'global_emb_to_decoder_map',
    'decoder_weight_scale': DECODER_WEIGHT_SCALE,
    'decoder_map_residual': map_residual,
    'decoder_bias_init': f'vietnamese_unigram_logfreq_{BIAS_COUNT_DOCS}docs',
    'special_token_init': 'copied_from_neobert_specials',
    'translator': TRANSLATOR_PATH,
    'fasttext_bin': str(FASTTEXT_BIN),
})

print(f'\nSaved embedding tensor : {EMBEDDING_TENSOR}')
print(f'Saved model/tokenizer  : {MODEL_DIR}')
print(f'\nTraining notebooks should reference: BASE_MODEL_REF = "init/{INIT_NAME}/model"')


── Re-injecting anchor/special embedding rows from live NeoBERT ──
  4990 anchor rows injected
  5 special rows injected

── Embedding norm calibration ──
  NeoBERT emb mean norm         : 1.2064
  SALT non-anchor mean (before) : 0.4030
  SALT non-anchor max  (before) : 1.2116
  SALT non-anchor mean (after)  : 1.2064
  SALT non-anchor std  (after)  : 0.000000

── Decoder init: global NeoBERT emb->dec map ──
  global emb->dec relative residual on NeoBERT: 0.549 (OK — linear head exists)
  decoder weight scale          : 0.1
  decoder mean row norm         : 0.1320

── Decoder bias: Vietnamese unigram log-frequency (20000 CulturaX docs) ──
  NeoBERT decoder bias norm (was zeroed before): 229.45


README.md:   0%|          | 0.00/32.6k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/90 [00:00<?, ?it/s]

  VI freq bias range            : [-16.78, -3.28]
  tokens counted                : 19,356,378

Embedding tensor saved: /content/drive/MyDrive/SALT3/init/videberta_salt_init_v5_globalmap_freqbias/init_vietnamese_embeddings.pt
Injecting embeddings and decoder into NeoBERT architecture...
LM head decoder: (30522, 768)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model + tokenizer saved: /content/drive/MyDrive/SALT3/init/videberta_salt_init_v5_globalmap_freqbias/model

── Safetensors integrity check ──
  Keys in file: 172
  model.encoder.weight present: True
  decoder.weight present      : True
  Max |safetensors - SALT emb| = 0.00e+00  ✓ OK
  Max |safetensors - decoder|  = 0.00e+00  ✓ OK

Saved embedding tensor : /content/drive/MyDrive/SALT3/init/videberta_salt_init_v5_globalmap_freqbias/init_vietnamese_embeddings.pt
Saved model/tokenizer  : /content/drive/MyDrive/SALT3/init/videberta_salt_init_v5_globalmap_freqbias/model

Training notebooks should reference: BASE_MODEL_REF = "init/videberta_salt_init_v5_globalmap_freqbias/model"


## Verify Saved Artifact

Reload the saved model and verify: embedding round-trip fidelity, tokenizer sanity, and forward pass produces finite logits.

In [12]:
import random

print('Reloading saved checkpoint for verification...')
reloaded = load_model_safe(MODEL_DIR, model_cls=AutoModelForMaskedLM, device=DEVICE)
reloaded_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

# ── Embedding round-trip fidelity ─────────────────────────────────────────────
loaded_emb = extract_embedding_weight(reloaded).float().cpu()
max_diff = (loaded_emb - new_embeddings.cpu()).abs().max().item()
print(f'Max |saved - loaded| embedding: {max_diff:.2e}  {"✓ OK" if max_diff < 1e-5 else "✗ MISMATCH!"}')
assert max_diff < 1e-5, f'Embedding round-trip error too large: {max_diff}'

# ── Decoder + bias round-trip fidelity ────────────────────────────────────────
loaded_dec = reloaded.decoder.weight.detach().float().cpu()
dec_diff = (loaded_dec - new_decoder.cpu()).abs().max().item()
print(f'Max |saved - loaded| decoder  : {dec_diff:.2e}  {"✓ OK" if dec_diff < 1e-5 else "✗ MISMATCH!"}')
assert dec_diff < 1e-5, f'Decoder round-trip error too large: {dec_diff}'

loaded_bias = reloaded.decoder.bias.detach().float().cpu()
bias_diff = (loaded_bias - new_decoder_bias.cpu()).abs().max().item()
print(f'Max |saved - loaded| dec bias : {bias_diff:.2e}  {"✓ OK" if bias_diff < 1e-5 else "✗ MISMATCH!"}')
assert bias_diff < 1e-5, f'Decoder bias round-trip error too large: {bias_diff}'

# ── Decoder is independent (not tied to embedding) ───────────────────────────
emb_dec_diff = (loaded_dec - loaded_emb).abs().max().item()
print(f'Max |decoder - embedding|     : {emb_dec_diff:.2e}  {"✓ untied" if emb_dec_diff > 1e-3 else "⚠ nearly identical"}')

# ── Embedding fingerprint ─────────────────────────────────────────────────────
fp = embedding_fingerprint(reloaded)
write_json(INIT_DIR / 'init_embedding_fingerprint.json', fp)
print('Embedding fingerprint:', fp)

# ── Special-token EMBEDDING rows must match NeoBERT exactly (decoder is mapped) ─
for row in special_token_pairs:
    emb_diff = (loaded_emb[row['target_id']] - source_embeddings[row['source_id']].cpu()).norm().item()
    print(f"  Special {row['key']}: target {row['target_id']} {row['target_token']} <- "
          f"source {row['source_id']} {row['source_token']} | emb L2={emb_diff:.4e}")
    assert emb_diff < 1e-6, f"Special token embedding mismatch for {row['key']}: {emb_diff}"

# ── Anchor spot-check: 10 random anchor EMBEDDING rows must match NeoBERT ─────
anchor_items = list(anchor_map.items())
random.shuffle(anchor_items)
anchor_ok = True
for vi_token, neo_token in anchor_items[:10]:
    vi_id = target_vocab[vi_token]
    neo_id = source_vocab[neo_token]
    emb_diff = (loaded_emb[vi_id] - source_embeddings[neo_id].cpu()).norm().item()
    if emb_diff > 1e-4:
        print(f'  ✗ Anchor "{vi_token}" -> "{neo_token}": emb L2={emb_diff:.4f}')
        anchor_ok = False
if anchor_ok:
    print('✓ 10 random anchor embedding rows match NeoBERT exactly')

# ── Forward pass sanity ───────────────────────────────────────────────────────
examples = ['Xin chào Việt Nam', 'Mô hình ngôn ngữ học tiếng Việt.']
for text in examples:
    batch = reloaded_tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = reloaded(**batch)
    assert torch.isfinite(out.logits).all(), f'Non-finite logits for: {text}'
    print(f'  "{text}" -> logits {tuple(out.logits.shape)} OK')

mask_text = f"Xin chào {reloaded_tokenizer.mask_token} Nam"
mask_batch = reloaded_tokenizer(mask_text, return_tensors='pt').to(DEVICE)
mask_pos = (mask_batch['input_ids'][0] == reloaded_tokenizer.mask_token_id).nonzero(as_tuple=False).squeeze(-1)
assert len(mask_pos) == 1, 'Expected exactly one mask token in verification probe.'
with torch.no_grad():
    mask_out = reloaded(**mask_batch)
mask_logits = mask_out.logits[0, mask_pos.item()]
mask_top_ids = mask_logits.topk(5).indices.cpu().tolist()
mask_top_tokens = reloaded_tokenizer.convert_ids_to_tokens(mask_top_ids)
print(f'  Mask probe "{mask_text}" -> top tokens: {mask_top_tokens}')

del reloaded
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('\nSALT init artifact is ready.')
print(f'Training notebooks should use: BASE_MODEL_REF = "init/{INIT_NAME}/model"')


Reloading saved checkpoint for verification...
  ✓ load_model_safe: all 172 keys loaded successfully
Max |saved - loaded| embedding: 0.00e+00  ✓ OK
Max |saved - loaded| decoder  : 0.00e+00  ✓ OK
Max |saved - loaded| dec bias : 0.00e+00  ✓ OK
Max |decoder - embedding|     : 6.07e-01  ✓ untied
Embedding fingerprint: {'shape': [30522, 768], 'mean_norm': 1.218300223350525, 'std_norm': 0.038181208074092865, 'sha256_first_rows': 'c8602bf4b19c04630c743797fdaf941a19a10d6c01e07220941e9a378e9978f2'}
  Special pad_token: target 0 [PAD] <- source 0 [PAD] | emb L2=0.0000e+00
  Special unk_token: target 3 [UNK] <- source 100 [UNK] | emb L2=0.0000e+00
  Special cls_token: target 1 [CLS] <- source 101 [CLS] | emb L2=0.0000e+00
  Special sep_token: target 2 [SEP] <- source 102 [SEP] | emb L2=0.0000e+00
  Special mask_token: target 4 [MASK] <- source 103 [MASK] | emb L2=0.0000e+00
✓ 10 random anchor embedding rows match NeoBERT exactly
  "Xin chào Việt Nam" -> logits (1, 6, 30522) OK
  "Mô hình ngôn ngữ

## Health Check — gate before CPT

Reloads the saved artifact and prints a single PASS/FAIL verdict so you know the re-init is safe to take to CPT.

In [13]:
# ── Final health gate: reload the saved artifact and verify it is CPT-ready ────
import math

print('Reloading saved artifact for health check...')
hc_model = load_model_safe(MODEL_DIR, model_cls=AutoModelForMaskedLM, device=DEVICE).eval()
hc_tok = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

vocab = hc_model.config.vocab_size
random_loss = math.log(vocab)
LOSS_GATE = 9.5  # step-0 MLM loss gate: near VI unigram entropy (~7.3), below random (~10.3)
checks = []  # (name, ok, detail)

# 1. tokenizer/model vocab agree
ok = len(hc_tok) == vocab
checks.append(('vocab size tokenizer==model', ok, f'{len(hc_tok)} vs {vocab}'))

# 2. embedding finite + sane norm
emb = extract_embedding_weight(hc_model).float().cpu()
emb_norm = emb.norm(dim=1).mean().item()
ok = torch.isfinite(emb).all().item() and 0.5 <= emb_norm <= 2.0
checks.append(('embedding finite + norm in [0.5,2.0]', ok, f'mean norm={emb_norm:.3f}'))

# 3. encoder layers are trained (not random-init)
rnd, tot = 0, 0
for name, p in hc_model.named_parameters():
    if 'transformer_encoder' in name or 'model.layers.' in name:
        tot += 1
        s = p.detach().float().std().item()
        if abs(s - 0.02) < 0.005 or s < 0.005:
            rnd += 1
ok = tot > 0 and rnd <= tot * 0.5
checks.append(('encoder weights trained (not random)', ok, f'{rnd}/{tot} look random'))

# 4. decoder weight finite + scaled (>0)
dec_w = hc_model.decoder.weight.detach().float().cpu()
dec_norm = dec_w.norm(dim=1).mean().item()
ok = torch.isfinite(dec_w).all().item() and dec_norm > 1e-3
checks.append(('decoder weight finite + non-zero', ok, f'mean row norm={dec_norm:.4f}'))

# 5. decoder bias = frequency prior present (NOT zeroed)
dec_b = hc_model.decoder.bias.detach().float().cpu()
ok = torch.isfinite(dec_b).all().item() and dec_b.norm().item() > 5.0
checks.append(('decoder bias = freq prior present', ok, f'norm={dec_b.norm():.2f} range=[{dec_b.min():.2f},{dec_b.max():.2f}]'))

# 6. forward pass finite + 7. quick masked-LM loss on Vietnamese
hc_sents = ['Việt Nam là một quốc gia ở Đông Nam Á.',
            'Hôm nay thời tiết rất đẹp và trời trong xanh.',
            'Kinh tế Việt Nam tăng trưởng mạnh trong năm qua.',
            'Cô ấy đọc sách trong thư viện mỗi buổi chiều.',
            'Bóng đá là môn thể thao được nhiều người yêu thích.']
enc = hc_tok(hc_sents, padding=True, truncation=True, max_length=48, return_tensors='pt').to(DEVICE)
ids = enc['input_ids']
torch.manual_seed(0)
special = set(hc_tok.all_special_ids)
probmat = torch.full(ids.shape, 0.20)
for sid in special:
    probmat[ids.cpu() == sid] = 0.0
masked = torch.bernoulli(probmat).bool().to(DEVICE)
labels = torch.full_like(ids, -100)
labels[masked] = ids[masked]
masked_ids = ids.clone()
masked_ids[masked] = hc_tok.mask_token_id
with torch.no_grad():
    out = hc_model(input_ids=masked_ids, attention_mask=enc['attention_mask'])
logits = out.logits
forward_ok = torch.isfinite(logits).all().item()
checks.append(('forward pass finite', forward_ok, 'ok' if forward_ok else 'NaN/Inf'))
mlm_loss = F.cross_entropy(logits.reshape(-1, vocab).float(), labels.reshape(-1), ignore_index=-100).item()
loss_ok = forward_ok and mlm_loss < LOSS_GATE
checks.append(('step-0 MLM loss < gate (9.5)', loss_ok,
               f'loss={mlm_loss:.2f} vs random={random_loss:.2f}'))

# ── verdict ───────────────────────────────────────────────────────────────────
print('\n── Init health check ──')
for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:38s} {detail}")
print(f"\n  Vietnamese unigram entropy ~7.3 is the expected step-0 band (bias-dominated).")
print(f"  Measured step-0 MLM loss (short VI sample): {mlm_loss:.2f}")

all_ok = all(ok for _, ok, _ in checks)
if all_ok and mlm_loss < LOSS_GATE:
    print('\n✅ INIT HEALTHY — safe to proceed to CPT (notebook 02).')
elif all_ok:
    print('\n⚠ STRUCTURE OK but step-0 loss high (>9.5). Re-check DECODER_WEIGHT_SCALE / bias before CPT.')
else:
    print('\n❌ NOT HEALTHY — fix the FAIL rows above before CPT. Do NOT spend GPU on this artifact.')

del hc_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Reloading saved artifact for health check...
  ✓ load_model_safe: all 172 keys loaded successfully

── Init health check ──
  [PASS] vocab size tokenizer==model            30522 vs 30522
  [PASS] embedding finite + norm in [0.5,2.0]   mean norm=1.218
  [PASS] encoder weights trained (not random)   13/168 look random
  [PASS] decoder weight finite + non-zero       mean row norm=0.1320
  [PASS] decoder bias = freq prior present      norm=2548.64 range=[-16.78,-3.28]
  [PASS] forward pass finite                    ok
  [PASS] step-0 MLM loss < gate (9.5)           loss=7.36 vs random=10.33

  Vietnamese unigram entropy ~7.3 is the expected step-0 band (bias-dominated).
  Measured step-0 MLM loss (short VI sample): 7.36

✅ INIT HEALTHY — safe to proceed to CPT (notebook 02).


In [14]:
import subprocess, time
from google.colab import runtime
subprocess.call(['sync'])
# time.sleep(30)
subprocess.call(['sync'])
print("Terminating runtime…")
runtime.unassign()

Terminating runtime…
